## Setup

In [1]:
!git clone --branch 3-condition-evaluation https://github.com/GoharKhachatryan/GenAI_project.git

Cloning into 'GenAI_project'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 206 (delta 119), reused 85 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 752.76 KiB | 2.80 MiB/s, done.
Resolving deltas: 100% (119/119), done.


In [2]:
%cd GenAI_project

/content/GenAI_project


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ln -s /content/drive/MyDrive/for_bostongene/data /content/GenAI_project/data
!ln -s /content/drive/MyDrive/for_bostongene/dataset.pt /content/GenAI_project/dataset.pt
!ln -s /content/drive/MyDrive/for_bostongene/pca_cache.npz /content/GenAI_project/pca_cache.npz

In [5]:
%cd GenAI_project

[Errno 2] No such file or directory: 'GenAI_project'
/content/GenAI_project


## Condition evaluation

This notebook performs the following operations:

1. Generates samples from the held-out condition sets (the test set).

2. Extracts features from the genetated samples.

3. Analyzes the distribution of the $||c - \hat{c}||$.

4. Performs a feature-level analysis.

## Import

In [6]:
import random
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from prepare_dataset import PairedCIFAR10
from pseudo_crossmodal import (
    extract_condition,
    load_pca,
)

from scripts.unet import UNet
from scripts.blocks import (
    ResBlock,
    ScaleShiftResBlock,
)
from scripts.model import (
    ConditionalDenoiser,
    ConditionalDenoiserConcat,
)
from scripts.diffusion import DDPM
from scripts.samplers import (
    ddpm_sample_respaced,
    ddim_sample,
)

## Reproducibility & Configs

In [7]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [8]:
NUM_STEPS = 100
BATCH_SIZE = 128

DDIM_ETA = 0.0

In [9]:
CHECKPOINTS_PATH = "/content/drive/MyDrive/for_bostongene/checkpoints/"
CHECKPOINTS = {
    "add_add": CHECKPOINTS_PATH + "add_add/" + "best_model.pt",
    "concat_add": CHECKPOINTS_PATH + "concat_add/" + "best_model.pt",
    "add_scale_shift": CHECKPOINTS_PATH + "add_scale_shift/" + "best_model.pt",
    "concat_scale_shift": CHECKPOINTS_PATH + "concat_scale_shift/" + "best_model.pt",
}

## Test set

In [10]:
test_dataset = PairedCIFAR10("test")

print("Test set size:", len(test_dataset))

image, cond, label = test_dataset[0]

print("Image shape:", image.shape)
print("Condition shape:", cond.shape)
print("Label:", label)

Test set size: 10000
Image shape: torch.Size([3, 32, 32])
Condition shape: torch.Size([16])
Label: tensor(3)


In [11]:
eval_indices = list(range(len(test_dataset)))
NUM_EVAL_SAMPLES = len(eval_indices)

print("Number of evaluation samples:", len(eval_indices))

Number of evaluation samples: 10000


In [12]:
generator = torch.Generator().manual_seed(SEED)

perm = torch.randperm(
    len(eval_indices),
    generator=generator,
).tolist()

eval_indices = [
    eval_indices[i]
    for i in perm
]

## Preprocessing

In [13]:
eval_images = []
eval_conditions = []
eval_labels = []

for idx in eval_indices:
    image, cond, label = test_dataset[idx]

    eval_images.append(image)
    eval_conditions.append(cond)
    eval_labels.append(label)

eval_images = torch.stack(eval_images)
eval_conditions = torch.stack(eval_conditions)
eval_labels = torch.tensor(eval_labels)

print(eval_images.shape)
print(eval_conditions.shape)
print(eval_labels.shape)

torch.Size([10000, 3, 32, 32])
torch.Size([10000, 16])
torch.Size([10000])


In [14]:
reference_checkpoint = torch.load(
    CHECKPOINTS["add_add"],
    map_location="cpu",
)

cond_mean = reference_checkpoint["cond_mean"]
cond_std = reference_checkpoint["cond_std"]

print(cond_mean.shape)
print(cond_std.shape)

torch.Size([16])
torch.Size([16])


In [15]:
eval_conditions_norm = (
    eval_conditions - cond_mean
) / (cond_std + 1e-8)

In [16]:
pca_state = load_pca()

In [17]:
generator = torch.Generator().manual_seed(SEED)

# Creats a fixed initial noise for each sample for fair evaluation
x_init_all = torch.randn(
    len(eval_indices),
    3,
    32,
    32,
    generator=generator,
)

## Model

In [18]:
# Re-use this function instead of manually creating the architecture for each experiment (4)
def build_model(variant):
    if variant == "add_add":
        unet = UNet(
            in_channels=3,
            out_channels=3,
            base_channels=64,
            emb_dim=256,
            block_cls=ResBlock,
        )

        model = ConditionalDenoiser(
            unet=unet,
            time_dim=256,
            cond_dim=16,
            emb_dim=256,
        )

    elif variant == "concat_add":
        unet = UNet(
            in_channels=3,
            out_channels=3,
            base_channels=64,
            emb_dim=256,
            block_cls=ResBlock,
        )

        model = ConditionalDenoiserConcat(
            unet=unet,
            time_dim=256,
            cond_dim=16,
            emb_dim=256,
        )

    elif variant == "add_scale_shift":
        unet = UNet(
            in_channels=3,
            out_channels=3,
            base_channels=64,
            emb_dim=256,
            block_cls=ScaleShiftResBlock,
        )

        model = ConditionalDenoiser(
            unet=unet,
            time_dim=256,
            cond_dim=16,
            emb_dim=256,
        )

    elif variant == "concat_scale_shift":
        unet = UNet(
            in_channels=3,
            out_channels=3,
            base_channels=64,
            emb_dim=256,
            block_cls=ScaleShiftResBlock,
        )

        model = ConditionalDenoiserConcat(
            unet=unet,
            time_dim=256,
            cond_dim=16,
            emb_dim=256,
        )

    else:
        raise ValueError(
            f"Unknown model variant: {variant}"
        )

    return model

In [19]:
def load_model(variant, device):
    model = build_model(variant)

    checkpoint = torch.load(
        CHECKPOINTS[variant],
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(device)
    model.eval()

    return model

In [20]:
diffusion = DDPM(
    timesteps=1000,
    schedule="cosine",
    device=device,
)

## Cycle metrics

In [21]:
def compute_cycle_metrics(
    cond_true,
    cond_hat,
    cond_std,
):
    error = cond_true - cond_hat

    raw_l2 = torch.linalg.vector_norm(
        error,
        dim=1,
    )

    standardized_error = (
        error
        / (cond_std + 1e-8)
    )

    standardized_l2 = torch.linalg.vector_norm(
        standardized_error,
        dim=1,
    )

    feature_mae = (
        error.abs().mean(dim=0)
    )

    feature_mae_std = (
        standardized_error.abs().mean(dim=0)
    )

    return {
        "raw_l2": raw_l2,
        "standardized_l2": standardized_l2,
        "feature_mae": feature_mae,
        "feature_mae_std": feature_mae_std,
    }

## Evaluation

In [22]:
@torch.no_grad()
def evaluate_cycle_consistency(
    model,
    sampler_name,
    cond_original,
    cond_normalized,
    x_init_all,
    cond_std,
    pca_state,
    diffusion,
    batch_size=128,
    num_steps=100,
    eta=0.0,
):
    all_cond_hat = []

    start_time = time.time()

    for start in range(
        0,
        len(cond_original),
        batch_size,
    ):
        end = min(
            start + batch_size,
            len(cond_original),
        )

        cond_batch = (
            cond_normalized[start:end]
            .to(device)
        )

        x_init_batch = (
            x_init_all[start:end]
            .to(device)
        )

        if sampler_name == "ddim":
            generated = ddim_sample(
                model=model,
                ddpm=diffusion,
                cond=cond_batch,
                num_steps=num_steps,
                eta=eta,
                x_init=x_init_batch,
            )

        elif sampler_name == "ddpm":
            # Resetting globally for every batch is NOT what we want.
            # Seed once before the evaluation loop instead.
            generated = ddpm_sample_respaced(
                model=model,
                ddpm=diffusion,
                cond=cond_batch,
                num_steps=num_steps,
                x_init=x_init_batch,
            )

        else:
            raise ValueError(
                f"Unknown sampler: {sampler_name}"
            )

        cond_hat = extract_condition(
            generated,
            pca_state,
        )

        all_cond_hat.append(
            cond_hat.cpu()
        )

    cond_hat = torch.cat(
        all_cond_hat,
        dim=0,
    )

    metrics = compute_cycle_metrics(
        cond_true=cond_original,
        cond_hat=cond_hat,
        cond_std=cond_std,
    )

    metrics["cond_hat"] = cond_hat
    metrics["runtime_sec"] = time.time() - start_time

    return metrics

## Experiment config

In [23]:
# Since the colab runtime can be interrupted, save the results after each update here
RESULTS_PATH = "/content/drive/MyDrive/for_bostongene/cycle_results_full_test.pt"

def save_results(results, path=RESULTS_PATH):
    torch.save(results, path)

In [24]:
import os

if os.path.exists(RESULTS_PATH):
    results = torch.load(
        RESULTS_PATH,
        map_location="cpu",
    )
    print(f"Loaded {len(results)} existing results.")
else:
    results = {}

In [25]:
def build_summary(results):
    rows = []

    for (variant, sampler), result in results.items():
        rows.append({
            "variant": variant,
            "sampler": sampler,
            "mean_raw_l2": result["raw_l2"].mean().item(),
            "std_raw_l2": result["raw_l2"].std().item(),
            "mean_standardized_l2":
                result["standardized_l2"].mean().item(),
            "std_standardized_l2":
                result["standardized_l2"].std().item(),
            "runtime_sec": result["runtime_sec"],
        })

    return pd.DataFrame(rows)

## A loop version

In [26]:
MODEL_VARIANTS = [
    "add_add",
    "concat_add",
    "add_scale_shift",
    "concat_scale_shift",
]

SAMPLERS = [
    "ddpm",
    "ddim",
]

In [27]:
for variant in MODEL_VARIANTS:
    print(f"\nLoading {variant}")

    model = load_model(variant, device)

    for sampler_name in SAMPLERS:

        key = (variant, sampler_name)

        # Skip already-computed configurations
        if key in results:
            print(f"  Skipping {key}: already computed.")
            continue

        print(
            f"  Evaluating {sampler_name}..."
        )

        # Reset RNG once before this complete evaluation
        torch.manual_seed(SEED)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        result = evaluate_cycle_consistency(
            model=model,
            sampler_name=sampler_name,
            cond_original=eval_conditions,
            cond_normalized=eval_conditions_norm,
            x_init_all=x_init_all,
            cond_std=cond_std,
            pca_state=pca_state,
            diffusion=diffusion,
            batch_size=BATCH_SIZE,
            num_steps=NUM_STEPS,
            eta=DDIM_ETA,
        )

        results[key] = result

        # Save immediately after successful completion
        save_results(results)

        print(
            "    standardized L2:",
            result["standardized_l2"]
            .mean()
            .item()
        )

        print(
            "    runtime:",
            round(
                result["runtime_sec"],
                1,
            ),
            "sec",
        )

        print("    Results saved.")

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Loading add_add
  Evaluating ddpm...
    standardized L2: 0.3508899211883545
    runtime: 439.4 sec
    Results saved.
  Evaluating ddim...
    standardized L2: 0.5585135817527771
    runtime: 443.1 sec
    Results saved.

Loading concat_add
  Evaluating ddpm...
    standardized L2: 0.35873958468437195
    runtime: 443.8 sec
    Results saved.
  Evaluating ddim...
    standardized L2: 0.4629572331905365
    runtime: 443.0 sec
    Results saved.

Loading add_scale_shift
  Evaluating ddpm...
    standardized L2: 0.2457181215286255
    runtime: 451.4 sec
    Results saved.
  Evaluating ddim...
    standardized L2: 0.4007309079170227
    runtime: 451.3 sec
    Results saved.

Loading concat_scale_shift
  Evaluating ddpm...
    standardized L2: 0.30169641971588135
    runtime: 452.9 sec
    Results saved.
  Evaluating ddim...
    standardized L2: 0.4143257737159729
    runtime: 452.8 sec
    Results saved.


## Build a summary from csv

In [28]:
if os.path.exists(RESULTS_PATH):
    results = torch.load(
        RESULTS_PATH,
        map_location="cpu",
    )
    print(f"Loaded {len(results)} existing results.")

Loaded 8 existing results.


In [29]:
summary = build_summary(results)

summary.to_csv(
    "/content/drive/MyDrive/for_bostongene/cycle_summary_full_test.csv",
    index=False,
)

## Plotting

In [30]:
feature_names = [
    "row_q1", "row_q2", "row_q3", "row_q4",
    "col_q1", "col_q2", "col_q3", "col_q4",
    "mean_R", "mean_G", "mean_B",
    "luma_std",
    "PCA_1", "PCA_2", "PCA_3", "PCA_4",
]

feature_rows = []

for (variant, sampler), result in results.items():
    for i, feature_name in enumerate(feature_names):
        feature_rows.append({
            "variant": variant,
            "sampler": sampler,
            "feature": feature_name,
            "raw_mae": result["feature_mae"][i].item(),
            "standardized_mae":
                result["feature_mae_std"][i].item(),
        })

feature_df = pd.DataFrame(feature_rows)

feature_df.head()

,variant,sampler,feature,raw_mae,standardized_mae
0,add_add,ddpm,row_q1,0.350394,0.052696
1,add_add,ddpm,row_q2,0.325972,0.072824
2,add_add,ddpm,row_q3,0.337772,0.083453
3,add_add,ddpm,row_q4,0.423422,0.079416
4,add_add,ddpm,col_q1,0.336020,0.065917


In [31]:
rows = []

for (variant, sampler), result in results.items():
    values = result["standardized_l2"]

    rows.append({
        "variant": variant,
        "sampler": sampler,
        "mean": values.mean().item(),
        "std": values.std().item(),
        "median": values.median().item(),
        "q25": torch.quantile(values, 0.25).item(),
        "q75": torch.quantile(values, 0.75).item(),
    })

distribution_summary = pd.DataFrame(rows)
distribution_summary

,variant,sampler,mean,std,median,q25,q75
0,add_add,ddpm,0.350890,0.108572,0.337580,0.273414,0.413243
1,add_add,ddim,0.558514,0.192617,0.540177,0.417328,0.681984
2,concat_add,ddpm,0.358740,0.118604,0.342911,0.273798,0.425942
3,concat_add,ddim,0.462957,0.153994,0.438678,0.354529,0.541972
4,add_scale_shift,ddpm,0.245718,0.075785,0.236535,0.192713,0.287466
5,add_scale_shift,ddim,0.400731,0.137760,0.383435,0.301745,0.482393
6,concat_scale_shift,ddpm,0.301696,0.098483,0.287401,0.232145,0.356131
7,concat_scale_shift,ddim,0.414326,0.135279,0.394100,0.316328,0.491788
